# Day 4: Model Training - Linear Regression

Building and training a Linear Regression model to predict student scores.

## 📋 Objectives
- Prepare data for modeling (train/test split)
- Train Linear Regression model
- Evaluate model performance
- Save model for deployment

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

# Styling
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('notebook', font_scale=1.1)

print("✅ Libraries imported")

In [ ]:
# Load prepared data
X = pd.read_csv('../../day12/data/features_scaled.csv')
y = pd.read_csv('../../day12/data/target.csv').squeeze()

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Feature columns: {list(X.columns)}")

In [ ]:
# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=True
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"Features: {X_train.shape[1]}")

In [ ]:
# Train Linear Regression Model
model = LinearRegression()
model.fit(X_train, y_train)

print("✅ Model trained successfully")
print(f"Intercept: {model.intercept_:.4f}")
print(f"Coefficients:")
for feat, coef in zip(X.columns, model.coef_):
    print(f"  {feat}: {coef:.4f}")

In [ ]:
# Predictions
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

# Training Metrics
train_mae = mean_absolute_error(y_train, y_train_pred)
train_mse = mean_squared_error(y_train, y_train_pred)
train_rmse = np.sqrt(train_mse)
train_r2 = r2_score(y_train, y_train_pred)

# Test Metrics
test_mae = mean_absolute_error(y_test, y_test_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
test_rmse = np.sqrt(test_mse)
test_r2 = r2_score(y_test, y_test_pred)

print("📊 TRAINING METRICS")
print(f"  MAE:  {train_mae:.4f}")
print(f"  MSE:  {train_mse:.4f}")
print(f"  RMSE: {train_rmse:.4f}")
print(f"  R²:   {train_r2:.4f}")

print("\n📊 TEST METRICS")
print(f"  MAE:  {test_mae:.4f}")
print(f"  MSE:  {test_mse:.4f}")
print(f"  RMSE: {test_rmse:.4f}")
print(f"  R²:   {test_r2:.4f}")

In [ ]:
# Cross-Validation
cv_scores = cross_val_score(model, X, y, cv=5, scoring='r2')
cv_mae = -cross_val_score(model, X, y, cv=5, scoring='neg_mean_absolute_error')
cv_mse = -cross_val_score(model, X, y, cv=5, scoring='neg_mean_squared_error')

print("🔄 5-Fold Cross-Validation Results:")
print(f"  R² scores: {cv_scores}")
print(f"  Mean R²: {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})")
print(f"  Mean MAE: {cv_mae.mean():.4f} (+/- {cv_mae.std()*2:.4f})")
print(f"  Mean MSE: {cv_mse.mean():.4f} (+/- {cv_mse.std()*2:.4f})")

In [ ]:
# Visualization: Actual vs Predicted
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training set
axes[0].scatter(y_train, y_train_pred, alpha=0.6, s=50, color='#2E86AB', label='Train')
axes[0].plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--', lw=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual Scores')
axes[0].set_ylabel('Predicted Scores')
axes[0].set_title(f'Training Set (R² = {train_r2:.3f})')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Test set
axes[1].scatter(y_test, y_test_pred, alpha=0.6, s=50, color='#A23B72', label='Test')
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Prediction')
axes[1].set_xlabel('Actual Scores')
axes[1].set_ylabel('Predicted Scores')
axes[1].set_title(f'Test Set (R² = {test_r2:.3f})')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Actual vs Predicted Scores', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../../day12/plots/actual_vs_predicted.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Residual Analysis
residuals_train = y_train - y_train_pred
residuals_test = y_test - y_test_pred

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Residuals vs Predicted (Train)
axes[0,0].scatter(y_train_pred, residuals_train, alpha=0.6, color='#2E86AB')
axes[0,0].axhline(y=0, color='r', linestyle='--', lw=2)
axes[0,0].set_xlabel('Predicted Scores')
axes[0,0].set_ylabel('Residuals')
axes[0,0].set_title('Residuals vs Predicted (Train)')
axes[0,0].grid(True, alpha=0.3)

# Residuals vs Predicted (Test)
axes[0,1].scatter(y_test_pred, residuals_test, alpha=0.6, color='#A23B72')
axes[0,1].axhline(y=0, color='r', linestyle='--', lw=2)
axes[0,1].set_xlabel('Predicted Scores')
axes[0,1].set_ylabel('Residuals')
axes[0,1].set_title('Residuals vs Predicted (Test)')
axes[0,1].grid(True, alpha=0.3)

# Residual Distribution (Train)
sns.histplot(residuals_train, kde=True, ax=axes[1,0], color='#2E86AB')
axes[1,0].axvline(x=0, color='r', linestyle='--', lw=2)
axes[1,0].set_xlabel('Residuals')
axes[1,0].set_title('Residual Distribution (Train)')

# Residual Distribution (Test)
sns.histplot(residuals_test, kde=True, ax=axes[1,1], color='#A23B72')
axes[1,1].axvline(x=0, color='r', linestyle='--', lw=2)
axes[1,1].set_xlabel('Residuals')
axes[1,1].set_title('Residual Distribution (Test)')

plt.suptitle('Residual Analysis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../../day12/plots/residual_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Feature Importance (Coefficients)
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_
}).sort_values('Coefficient', key=abs, ascending=False)

plt.figure(figsize=(10, 6))
colors = ['#2E86AB' if c > 0 else '#A23B72' for c in coef_df['Coefficient']]
bars = plt.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors)
plt.xlabel('Coefficient Value')
plt.title('Feature Coefficients (Linear Regression)', fontsize=14, fontweight='bold')
plt.axvline(x=0, color='black', linewidth=0.5)
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig('../../day12/plots/feature_coefficients.png', dpi=300, bbox_inches='tight')
plt.show()

display(coef_df)

In [ ]:
# Save Model and Metadata
import joblib

# Save model
joblib.dump(model, '../../day12/models/linear_regression_model.pkl')
print("✅ Model saved: linear_regression_model.pkl")

# Save metadata
metadata = {
    'model_type': 'LinearRegression',
    'features': list(X.columns),
    'target': 'score',
    'train_samples': int(X_train.shape[0]),
    'test_samples': int(X_test.shape[0]),
    'metrics': {
        'train': {'mae': float(train_mae), 'mse': float(train_mse), 'rmse': float(train_rmse), 'r2': float(train_r2)},
        'test': {'mae': float(test_mae), 'mse': float(test_mse), 'rmse': float(test_rmse), 'r2': float(test_r2)},
        'cv': {'mean_r2': float(cv_scores.mean()), 'std_r2': float(cv_scores.std())}
    },
    'coefficients': dict(zip(X.columns, model.coef_.tolist())),
    'intercept': float(model.intercept_),
    'random_state': 42
}

with open('../../day12/models/model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print("✅ Metadata saved: model_metadata.json")

## 📝 Summary

- Linear Regression model trained on scaled features
- Train/Test split: 80/20 with random_state=42
- Cross-validation (5-fold) performed for robust evaluation
- Residual analysis shows good model fit
- Feature coefficients interpreted for importance
- Model and metadata saved for deployment

---
*Next: [05_model_evaluation.ipynb](05_model_evaluation.ipynb)*